In [27]:
!pip install -q -U transformers accelerate gradio
!apt-get update -qq
!apt-get install -y -qq g++

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [28]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

else:
    raise RuntimeError(
        "GPU not detected. Enable T4 GPU in Colab."
    )

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [29]:
models = {

    "Qwen Coder 0.5B": {
        "id": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
        "parameters": "0.5B"
    },

    "Qwen Coder 1.5B": {
        "id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "parameters": "1.5B"
    },

    "DeepSeek Coder 1.3B": {
        "id": "deepseek-ai/deepseek-coder-1.3b-instruct",
        "parameters": "1.3B"
    },

    "Qwen Coder 3B": {
        "id": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "parameters": "3B"
    }

}

print("Registered models:")

for name, details in models.items():

    print(
        f"{name} | "
        f"{details['id']} | "
        f"{details['parameters']}"
    )

Registered models:
Qwen Coder 0.5B | Qwen/Qwen2.5-Coder-0.5B-Instruct | 0.5B
Qwen Coder 1.5B | Qwen/Qwen2.5-Coder-1.5B-Instruct | 1.5B
DeepSeek Coder 1.3B | deepseek-ai/deepseek-coder-1.3b-instruct | 1.3B
Qwen Coder 3B | Qwen/Qwen2.5-Coder-3B-Instruct | 3B


In [30]:
import torch
import gc
import time

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)


# Currently loaded model
current_model = None
current_tokenizer = None
current_model_name = None


def unload_model():

    global current_model
    global current_tokenizer
    global current_model_name

    if current_model is not None:
        del current_model

    if current_tokenizer is not None:
        del current_tokenizer

    current_model = None
    current_tokenizer = None
    current_model_name = None

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Previous model unloaded.")


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(model_id):

    print(f"Loading model: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        use_safetensors=True
    )

    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Model loaded successfully!")
    print("Device:", model.device)

    return tokenizer, model

    # Unload previous model
    unload_model()

    model_id = models[model_name]["id"]

    print("=" * 60)
    print("Loading:", model_name)
    print("Hugging Face ID:", model_id)
    print("=" * 60)

    remote_code = model_id.startswith("deepseek-ai/")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=remote_code
    )

    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=remote_code
    )

    model.eval()

    # Set padding token if missing
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Store globally
    current_model = model
    current_tokenizer = tokenizer
    current_model_name = model_name

    print("\nModel loaded successfully!")
    print("Model:", model_name)
    print("Device:", model.device)

    print(
        "GPU Memory Allocated:",
        round(
            torch.cuda.memory_allocated() / 1024**3,
            2
        ),
        "GB"
    )

    return tokenizer, model

In [31]:

import re
import time
import gc
import torch


# --------------------------------------------------
# MODEL CONFIGURATION
# --------------------------------------------------

MODELS = {
    "Qwen Coder 0.5B": {
        "id": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
        "parameters": "0.5B"
    },

    "Qwen Coder 1.5B": {
        "id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "parameters": "1.5B"
    },

    "DeepSeek Coder 1.3B": {
        "id": "deepseek-ai/deepseek-coder-1.3b-instruct",
        "parameters": "1.3B"
    },

    "Qwen Coder 3B": {
        "id": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "parameters": "3B"
    }
}


# --------------------------------------------------
# SYSTEM PROMPT
# --------------------------------------------------

SYSTEM_PROMPT = """
You are an expert Python-to-C++ code conversion engineer.

Your task is to convert Python programs into optimized C++17.

Requirements:

1. Produce complete, compilable C++17 programs.
2. Preserve the mathematical behavior of the Python program.
3. Preserve the original input/output behavior.
4. Use efficient algorithms and appropriate data structures.
5. Optimize execution speed without changing program behavior.
6. Use appropriate C++ data types.
7. Handle edge cases correctly.
8. Include all necessary C++ headers.
9. Do not introduce unnecessary computations.
10. Do not print execution time.
11. Do not provide explanations.

Return ONLY the complete C++17 code.
"""


# --------------------------------------------------
# PYTHON TO C++ GENERATOR
# --------------------------------------------------

def generate_cpp(python_code, model_name):

    if not isinstance(python_code, str) or not python_code.strip():
        raise ValueError("Python code cannot be empty.")

    # Resolve display name to Hugging Face model ID
    if model_name in MODELS:
        model_id = MODELS[model_name]["id"]
    else:
        # Allow direct Hugging Face model IDs too
        model_id = model_name

    print(f"Loading model: {model_id}")

    tokenizer, model = load_model(model_id)

    try:

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    "Convert the following Python program "
                    "into an equivalent optimized C++17 program.\n\n"
                    f"Python code:\n{python_code}"
                )
            }
        ]

        # Apply model-specific chat template
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        )

        # Find input embedding device
        input_device = model.get_input_embeddings().weight.device

        inputs = {
            key: value.to(input_device)
            for key, value in inputs.items()
        }

        # Synchronize GPU before timing
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=1600,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Synchronize GPU after generation
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        generation_time = time.perf_counter() - start_time

        # Remove original prompt tokens
        generated_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        # Decode generated tokens
        cpp_code = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # Remove Markdown code fences
        match = re.search(
            r"```(?:cpp|c\+\+)?\s*(.*?)```",
            cpp_code,
            flags=re.DOTALL | re.IGNORECASE
        )

        if match:
            cpp_code = match.group(1).strip()

        if not cpp_code:
            raise RuntimeError("Model returned empty C++ code.")

        print(f"Generation completed: {generation_time:.4f} seconds")

        return cpp_code, generation_time

    finally:

        # Release GPU memory before loading next model
        del model
        del tokenizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [32]:
sample_python = """
def calculate(n):

    result = 0

    for i in range(1, n + 1):
        result += i

    return result


print(calculate(100))
"""


cpp_code, generation_time = generate_cpp(
    sample_python,
    "Qwen Coder 0.5B"
)

print("GENERATED C++ CODE")
print("=" * 60)

print(cpp_code)

print("\nGENERATION METRICS")
print("=" * 60)

print("Model: Qwen Coder 0.5B")
print(f"Generation time: {generation_time:.3f} seconds")

Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 17.3932 seconds
GENERATED C++ CODE
#include <iostream>

int calculate(int n) {
    int result = 0;

    for (int i = 1; i <= n; ++i) {
        result += i;
    }

    return result;
}

int main() {
    int n = 100;
    int result = calculate(n);
    std::cout << "The result of calculate(" << n << ") is: " << result << std::endl;
    return 0;
}

GENERATION METRICS
Model: Qwen Coder 0.5B
Generation time: 17.393 seconds


In [33]:
# First model
tokenizer, model = load_model("Qwen Coder 0.5B")

print("\nCurrently loaded:", current_model_name)


# Request the same model again
tokenizer, model = load_model("Qwen Coder 0.5B")

print("\nCurrently loaded:", current_model_name)


# Switch to another model
tokenizer, model = load_model("Qwen Coder 1.5B")

print("\nCurrently loaded:", current_model_name)

Loading model: Qwen Coder 0.5B


OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'Qwen Coder 0.5B'.

In [44]:
print("MODELS:", MODELS)
print("Current model:", current_model_name)
print("Model object exists:", model is not None)
print("Tokenizer exists:", tokenizer is not None)

MODELS: {'Qwen Coder 0.5B': {'id': 'Qwen/Qwen2.5-Coder-0.5B-Instruct', 'parameters': '0.5B'}, 'Qwen Coder 1.5B': {'id': 'Qwen/Qwen2.5-Coder-1.5B-Instruct', 'parameters': '1.5B'}, 'DeepSeek Coder 1.3B': {'id': 'deepseek-ai/deepseek-coder-1.3b-instruct', 'parameters': '1.3B'}, 'Qwen Coder 3B': {'id': 'Qwen/Qwen2.5-Coder-3B-Instruct', 'parameters': '3B'}}
Current model: None
Model object exists: True
Tokenizer exists: True


In [45]:
# First model
tokenizer, model = load_model("Qwen Coder 0.5B")
print("Currently loaded:", current_model_name)

# Request the same model again
tokenizer, model = load_model("Qwen Coder 0.5B")
print("Currently loaded:", current_model_name)

# Switch to another model
tokenizer, model = load_model("Qwen Coder 1.5B")
print("Currently loaded:", current_model_name)

Loading model: Qwen Coder 0.5B


OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'Qwen Coder 0.5B'.

In [34]:
sample_python = """
def calculate(n):

    result = 0

    for i in range(1, n + 1):
        result += i

    return result


print(calculate(100))
"""


cpp_code, generation_time = generate_cpp(
    sample_python,
    "Qwen Coder 1.5B"
)

print("=" * 60)
print("GENERATED C++")
print("=" * 60)

print(cpp_code)

print("\nGeneration time:", round(generation_time, 3), "seconds")

Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 2.3003 seconds
GENERATED C++
#include <iostream>

int calculate(int n) {
    int result = 0;
    for (int i = 1; i <= n; ++i) {
        result += i;
    }
    return result;
}

int main() {
    std::cout << calculate(100) << std::endl;
    return 0;
}

Generation time: 2.3 seconds


In [35]:
import subprocess
import tempfile
import time
from pathlib import Path


def compile_and_run_cpp(
    cpp_code,
    input_data="",
    compile_timeout=120,
    execution_timeout=30
):
    """
    Compile generated C++17 code and execute it.

    Returns compilation status, execution status,
    stdout, stderr, and timing information.
    """

    results = {
        "compile_success": False,
        "compile_time": None,
        "compile_error": "",
        "execution_success": False,
        "execution_time": None,
        "stdout": "",
        "stderr": "",
        "return_code": None
    }

    with tempfile.TemporaryDirectory() as temp_dir:

        cpp_file = Path(temp_dir) / "main.cpp"
        executable = Path(temp_dir) / "main"

        # Save generated C++ code
        cpp_file.write_text(cpp_code, encoding="utf-8")

        # Compile using g++ and C++17
        compile_command = [
            "g++",
            "-std=c++17",
            "-O2",
            str(cpp_file),
            "-o",
            str(executable)
        ]

        start_time = time.perf_counter()

        try:
            compilation = subprocess.run(
                compile_command,
                capture_output=True,
                text=True,
                timeout=compile_timeout
            )

            results["compile_time"] = (
                time.perf_counter() - start_time
            )

            results["compile_success"] = (
                compilation.returncode == 0
            )

            results["compile_error"] = compilation.stderr

            if not results["compile_success"]:
                return results

        except subprocess.TimeoutExpired:
            results["compile_time"] = (
                time.perf_counter() - start_time
            )
            results["compile_error"] = "Compilation timed out."
            return results

        # Execute the compiled program
        start_time = time.perf_counter()

        try:
            execution = subprocess.run(
                [str(executable)],
                input=input_data,
                capture_output=True,
                text=True,
                timeout=execution_timeout
            )

            results["execution_time"] = (
                time.perf_counter() - start_time
            )

            results["stdout"] = execution.stdout
            results["stderr"] = execution.stderr
            results["return_code"] = execution.returncode

            results["execution_success"] = (
                execution.returncode == 0
            )

        except subprocess.TimeoutExpired:
            results["execution_time"] = (
                time.perf_counter() - start_time
            )
            results["stderr"] = "Execution timed out."

    return results

In [36]:
benchmark = compile_and_run_cpp(cpp_code)

print("=" * 50)
print("C++ COMPILATION RESULTS")
print("=" * 50)

print("Compilation successful:", benchmark["compile_success"])
print("Compilation time:", round(benchmark["compile_time"], 5), "seconds")

print("\nExecution successful:", benchmark["execution_success"])
print("Execution time:", round(benchmark["execution_time"], 5), "seconds")

print("\nProgram output:")
print(benchmark["stdout"])

if benchmark["compile_error"]:
    print("\nCompilation errors:")
    print(benchmark["compile_error"])

if benchmark["stderr"]:
    print("\nRuntime errors:")
    print(benchmark["stderr"])

C++ COMPILATION RESULTS
Compilation successful: True
Compilation time: 0.39907 seconds

Execution successful: True
Execution time: 0.0019 seconds

Program output:
5050



In [37]:
!pip install -q -U gradio

In [46]:

import gradio as gr
import time

def generate_handler(python_code, model_name):
    if not python_code.strip():
        return "", "Please enter Python code first."

    try:
        cpp_code, generation_time = generate_cpp(
            python_code,
            model_name
        )

        info = (
            f"Model: {model_name}\n"
            f"Generation time: {generation_time:.4f} seconds"
        )

        return cpp_code, info

    except Exception as e:
        return "", f"Generation failed: {str(e)}"


def compile_handler(cpp_code, input_data):
    if not cpp_code.strip():
        return "Please generate or enter C++ code first.", ""

    try:
        results = compile_and_run_cpp(
            cpp_code,
            input_data=input_data
        )

        report = (
            f"Compilation successful: {results['compile_success']}\n"
            f"Compilation time: {results['compile_time']}\n\n"
            f"Execution successful: {results['execution_success']}\n"
            f"Execution time: {results['execution_time']}\n"
            f"Return code: {results['return_code']}"
        )

        if results["compile_error"]:
            report += "\n\nCompilation error:\n" + results["compile_error"]

        if results["stderr"]:
            report += "\n\nRuntime error:\n" + results["stderr"]

        return report, results["stdout"]

    except Exception as e:
        return f"Execution failed: {str(e)}", ""


with gr.Blocks(title="Python to C++ AI Generator") as demo:

    gr.Markdown(
        """
        # Python → C++ AI Code Generator

        Convert Python code into optimized C++17 using Hugging Face models.
        """
    )

    model_dropdown = gr.Dropdown(
        choices=list(models.keys()),
        value="Qwen Coder 1.5B",
        label="Select AI Model"
    )

    python_input = gr.Code(
        label="Python Code",
        language="python",
        lines=12
    )

    generate_button = gr.Button(
        "Generate C++",
        variant="primary"
    )

    generation_info = gr.Textbox(
        label="Generation Information",
        interactive=False
    )

    cpp_output = gr.Code(
        label="Generated C++17 Code",
        language="cpp",
        lines=15
    )

    gr.Markdown("## Compile and Execute")

    input_data = gr.Textbox(
        label="Program Input (stdin)",
        placeholder="Enter input if your C++ program requires it...",
        lines=3
    )

    compile_button = gr.Button(
        "Compile and Run",
        variant="primary"
    )

    compile_info = gr.Textbox(
        label="Compilation and Execution Results",
        lines=8,
        interactive=False
    )

    program_output = gr.Textbox(
        label="Program Output (stdout)",
        lines=5,
        interactive=False
    )

    generate_button.click(
        fn=generate_handler,
        inputs=[python_input, model_dropdown],
        outputs=[cpp_output, generation_info]
    )

    compile_button.click(
        fn=compile_handler,
        inputs=[cpp_output, input_data],
        outputs=[compile_info, program_output]
    )


demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3e4f8c9878cf0cf9fc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 3.1980 seconds
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 8.3121 seconds
Loading model: deepseek-ai/deepseek-coder-1.3b-instruct
Loading model: deepseek-ai/deepseek-coder-1.3b-instruct


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 5.1474 seconds
Loading model: Qwen/Qwen2.5-Coder-3B-Instruct
Loading model: Qwen/Qwen2.5-Coder-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 6.9557 seconds
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3e4f8c9878cf0cf9fc.gradio.live


In [39]:

from huggingface_hub import snapshot_download
import os

models_to_download = {
    "DeepSeek Coder 1.3B": "deepseek-ai/deepseek-coder-1.3b-instruct",
    "Qwen Coder 3B": "Qwen/Qwen2.5-Coder-3B-Instruct"
}

# Download only the files required for Transformers inference.
# Avoid downloading duplicate PyTorch .bin weights.
allow_patterns = [
    "*.safetensors",
    "*.json",
    "*.txt",
    "*.model",
    "*.py",
    "README*",
    "LICENSE*"
]

for name, model_id in models_to_download.items():

    print("\n" + "=" * 60)
    print("Downloading:", name)
    print("Model ID:", model_id)
    print("=" * 60)

    try:
        local_path = snapshot_download(
            repo_id=model_id,
            allow_patterns=allow_patterns,
            ignore_patterns=["*.bin", "*.h5", "*.msgpack"]
        )

        print("\nDownload completed successfully!")
        print("Local path:", local_path)

        weight_files = [
            f for f in os.listdir(local_path)
            if f.endswith(".safetensors")
        ]

        print("Safetensors files:", weight_files)

    except Exception as e:
        print(f"\nDownload failed for {name}: {e}")
        raise

print("\nBoth model download operations completed!")



Downloading: DeepSeek Coder 1.3B
Model ID: deepseek-ai/deepseek-coder-1.3b-instruct


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


Download completed successfully!
Local path: /root/.cache/huggingface/hub/models--deepseek-ai--deepseek-coder-1.3b-instruct/snapshots/e063262dac8366fc1f28a4da0ff3c50ea66259ca
Safetensors files: ['model.safetensors']

Downloading: Qwen Coder 3B
Model ID: Qwen/Qwen2.5-Coder-3B-Instruct


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


Download completed successfully!
Local path: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5
Safetensors files: ['model-00002-of-00002.safetensors', 'model-00001-of-00002.safetensors']

Both model download operations completed!


In [40]:
from huggingface_hub import try_to_load_from_cache

model_ids = [
    "deepseek-ai/deepseek-coder-1.3b-instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct"
]

for model_id in model_ids:
    path = snapshot_download(
        repo_id=model_id,
        allow_patterns=["*.safetensors", "config.json"]
    )

    print(f"\n{model_id}")
    print("Cached path:", path)
    print("Verified:", os.path.exists(path))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]


deepseek-ai/deepseek-coder-1.3b-instruct
Cached path: /root/.cache/huggingface/hub/models--deepseek-ai--deepseek-coder-1.3b-instruct/snapshots/e063262dac8366fc1f28a4da0ff3c50ea66259ca
Verified: True


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]


Qwen/Qwen2.5-Coder-3B-Instruct
Cached path: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5
Verified: True


In [41]:
MODELS = {
    "Qwen Coder 0.5B": {
        "id": "Qwen/Qwen2.5-Coder-0.5B-Instruct",
        "parameters": "0.5B"
    },

    "Qwen Coder 1.5B": {
        "id": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "parameters": "1.5B"
    },

    "DeepSeek Coder 1.3B": {
        "id": "deepseek-ai/deepseek-coder-1.3b-instruct",
        "parameters": "1.3B"
    },

    "Qwen Coder 3B": {
        "id": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "parameters": "3B"
    }
}

In [42]:
from huggingface_hub import snapshot_download

for name, details in MODELS.items():

    print(f"\nChecking {name}...")

    path = snapshot_download(
        repo_id=details["id"],
        allow_patterns=[
            "*.safetensors",
            "*.json",
            "*.model",
            "*.txt"
        ]
    )

    print("Available:", path)


Checking Qwen Coder 0.5B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Available: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-0.5B-Instruct/snapshots/ea3f2471cf1b1f0db85067f1ef93848e38e88c25

Checking Qwen Coder 1.5B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

... (3 outputs left out)

Available: /root/.cache/huggingface/hub/models--deepseek-ai--deepseek-coder-1.3b-instruct/snapshots/e063262dac8366fc1f28a4da0ff3c50ea66259ca

Checking Qwen Coder 3B...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Available: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-3B-Instruct/snapshots/488639f1ff808d1d3d0ba301aef8c11461451ec5


In [43]:
for model_name in MODELS:

    print("\n" + "=" * 60)
    print("Testing:", model_name)
    print("=" * 60)

    cpp_code, generation_time = generate_cpp(
        sample_python,
        model_name
    )

    print("\nGenerated C++:")
    print(cpp_code)

    print("\nGeneration time:", generation_time)


Testing: Qwen Coder 0.5B
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 12.5760 seconds

Generated C++:
#include <iostream>

int calculate(int n) {
    int result = 0;

    for (int i = 1; i <= n; ++i) {
        result += i;
    }

    return result;
}

int main() {
    int n = 100;
    int result = calculate(n);
    std::cout << "The result of calculate(" << n << ") is: " << result << std::endl;
    return 0;
}

Generation time: 12.575992956999926

Testing: Qwen Coder 1.5B
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 5.5940 seconds

Generated C++:
#include <iostream>

int calculate(int n) {
    int result = 0;
    for (int i = 1; i <= n; ++i) {
        result += i;
    }
    return result;
}

int main() {
    std::cout << calculate(100) << std::endl;
    return 0;
}

Generation time: 5.5939689140000155

Testing: DeepSeek Coder 1.3B
Loading model: deepseek-ai/deepseek-coder-1.3b-instruct
Loading model: deepseek-ai/deepseek-coder-1.3b-instruct


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 4.4220 seconds

Generated C++:
#include <iostream>

int calculate(int n) {
    int result = 0;

    for(int i = 1; i <= n; i++) {
        result += i;
    }

    return result;
}

int main() {
    std::cout << calculate(100) << std::endl;

    return 0;
}

Generation time: 4.422021288999986

Testing: Qwen Coder 3B
Loading model: Qwen/Qwen2.5-Coder-3B-Instruct
Loading model: Qwen/Qwen2.5-Coder-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 6.9703 seconds

Generated C++:
#include <iostream>

int calculate(int n) {
    int result = 0;
    for (int i = 1; i <= n; ++i) {
        result += i;
    }
    return result;
}

int main() {
    std::cout << calculate(100) << std::endl;
    return 0;
}

Generation time: 6.970291784999972


In [47]:

import statistics


def analyze_performance(results):
    """
    Analyze execution-time measurements for multiple AI-generated
    C++ implementations.

    Expected format:
    [
        {
            "model": "Qwen Coder 0.5B",
            "execution_times": [0.0021, 0.0023, 0.0020],
            "compile_time": 0.45,
            "correct": True
        }
    ]
    """

    if not results:
        raise ValueError("No benchmark results provided.")

    analysis = []

    for result in results:

        model_name = result["model"]
        times = result["execution_times"]

        if not times:
            print(f"Skipping {model_name}: no execution measurements.")
            continue

        if not result.get("correct", True):
            print(f"Skipping {model_name}: incorrect output.")
            continue

        if any(t < 0 for t in times):
            raise ValueError(
                f"Negative execution time found for {model_name}."
            )

        stats = {
            "model": model_name,
            "runs": len(times),
            "average_time": statistics.mean(times),
            "median_time": statistics.median(times),
            "min_time": min(times),
            "max_time": max(times),
            "std_dev": (
                statistics.stdev(times)
                if len(times) > 1 else 0.0
            ),
            "compile_time": result.get("compile_time")
        }

        analysis.append(stats)

    if not analysis:
        raise ValueError("No valid benchmark results to analyze.")

    # Fastest implementation becomes the baseline
    fastest_time = min(
        item["median_time"] for item in analysis
    )

    for item in analysis:
        item["speedup_vs_fastest"] = (
            item["median_time"] / fastest_time
        )

    # Rank by median execution time
    analysis.sort(key=lambda x: x["median_time"])

    for rank, item in enumerate(analysis, start=1):
        item["rank"] = rank

    return analysis


def display_analysis(analysis):

    print("\n========== C++ PERFORMANCE ANALYSIS ==========\n")

    for item in analysis:
        print(f"Rank: {item['rank']}")
        print(f"Model: {item['model']}")
        print(f"Runs: {item['runs']}")
        print(f"Average: {item['average_time']:.6f} seconds")
        print(f"Median: {item['median_time']:.6f} seconds")
        print(f"Minimum: {item['min_time']:.6f} seconds")
        print(f"Maximum: {item['max_time']:.6f} seconds")
        print(f"Std deviation: {item['std_dev']:.6f} seconds")
        print(
            f"Relative to fastest: "
            f"{item['speedup_vs_fastest']:.2f}x"
        )
        print("-" * 45)

    winner = analysis[0]

    print("\nFASTEST IMPLEMENTATION")
    print(f"Model: {winner['model']}")
    print(f"Median execution time: {winner['median_time']:.6f}s")


In [48]:
results = [
    {
        "model": "Qwen Coder 0.5B",
        "execution_times": [0.0042, 0.0045, 0.0041, 0.0043, 0.0044],
        "compile_time": 0.45,
        "correct": True
    },
    {
        "model": "Qwen Coder 1.5B",
        "execution_times": [0.0028, 0.0030, 0.0029, 0.0027, 0.0029],
        "compile_time": 0.62,
        "correct": True
    },
    {
        "model": "DeepSeek Coder 1.3B",
        "execution_times": [0.0035, 0.0036, 0.0034, 0.0037, 0.0035],
        "compile_time": 0.51,
        "correct": True
    }
]

analysis = analyze_performance(results)
display_analysis(analysis)


========== C++ PERFORMANCE ANALYSIS ==========

Rank: 1
Model: Qwen Coder 1.5B
Runs: 5
Average: 0.002860 seconds
Median: 0.002900 seconds
Minimum: 0.002700 seconds
Maximum: 0.003000 seconds
Std deviation: 0.000114 seconds
Relative to fastest: 1.00x
---------------------------------------------
Rank: 2
Model: DeepSeek Coder 1.3B
Runs: 5
Average: 0.003540 seconds
Median: 0.003500 seconds
Minimum: 0.003400 seconds
Maximum: 0.003700 seconds
Std deviation: 0.000114 seconds
Relative to fastest: 1.21x
---------------------------------------------
Rank: 3
Model: Qwen Coder 0.5B
Runs: 5
Average: 0.004300 seconds
Median: 0.004300 seconds
Minimum: 0.004100 seconds
Maximum: 0.004500 seconds
Std deviation: 0.000158 seconds
Relative to fastest: 1.48x
---------------------------------------------

FASTEST IMPLEMENTATION
Model: Qwen Coder 1.5B
Median execution time: 0.002900s


In [49]:
def generate_all_models(python_code):

    generated_results = {}

    for model_name in MODELS:

        print(f"\nGenerating code using {model_name}...")

        try:
            cpp_code, generation_time = generate_cpp(
                python_code,
                model_name
            )

            generated_results[model_name] = {
                "cpp_code": cpp_code,
                "generation_time": generation_time
            }

            print(f"Generation successful: {generation_time:.4f}s")

        except Exception as e:
            print(f"Generation failed: {e}")

            generated_results[model_name] = {
                "cpp_code": None,
                "generation_time": None,
                "error": str(e)
            }

    return generated_results

In [50]:
def analyze_generated_code(generated_results, expected_output,
                           input_data="", runs=5):

    performance_results = []

    for model_name, generated in generated_results.items():

        print(f"\nAnalyzing: {model_name}")

        cpp_code = generated["cpp_code"]

        if not cpp_code:
            continue

        test_result = test_cpp_code(
            cpp_code=cpp_code,
            expected_output=expected_output,
            input_data=input_data,
            runs=runs
        )

        result = {
            "model": model_name,
            "generation_time": generated["generation_time"],
            "compile_time": test_result["compile_time"],
            "execution_times": test_result["execution_times"],
            "correct": (
                test_result["compile_success"]
                and test_result["correct"]
            ),
            "compile_success": test_result["compile_success"],
            "error": test_result["error"]
        }

        performance_results.append(result)

        print("Compilation:", test_result["compile_success"])
        print("Correctness:", result["correct"])
        print("Execution times:", result["execution_times"])

    return performance_results

In [51]:
def rank_models(performance_results):

    valid_results = [
        result for result in performance_results
        if result["compile_success"] and result["correct"]
        and result["execution_times"]
    ]

    if not valid_results:
        print("No models produced valid executable code.")
        return []

    rankings = analyze_performance(valid_results)

    display_analysis(rankings)

    return rankings

In [53]:

import subprocess
import tempfile
import os
import time


def test_cpp_code(cpp_code, expected_output, input_data="",
                  runs=5, timeout=10):

    result = {
        "compile_success": False,
        "correct": False,
        "compile_time": None,
        "execution_times": [],
        "output": None,
        "error": None
    }

    with tempfile.TemporaryDirectory() as temp_dir:

        cpp_file = os.path.join(temp_dir, "generated.cpp")
        executable = os.path.join(temp_dir, "generated")

        # Save C++ code
        with open(cpp_file, "w") as f:
            f.write(cpp_code)

        # Compile C++ code
        compile_start = time.perf_counter()

        try:
            compilation = subprocess.run(
                ["g++", "-std=c++17", "-O2",
                 cpp_file, "-o", executable],
                capture_output=True,
                text=True,
                timeout=timeout
            )

        except subprocess.TimeoutExpired:
            result["error"] = "Compilation timed out."
            return result

        result["compile_time"] = time.perf_counter() - compile_start

        if compilation.returncode != 0:
            result["error"] = compilation.stderr
            return result

        result["compile_success"] = True

        # Execute multiple times
        for _ in range(runs):

            try:
                start = time.perf_counter()

                execution = subprocess.run(
                    [executable],
                    input=input_data,
                    capture_output=True,
                    text=True,
                    timeout=timeout
                )

                elapsed = time.perf_counter() - start

            except subprocess.TimeoutExpired:
                result["error"] = "Execution timed out."
                result["execution_times"] = []
                return result

            if execution.returncode != 0:
                result["error"] = execution.stderr
                result["execution_times"] = []
                return result

            result["execution_times"].append(elapsed)
            result["output"] = execution.stdout.strip()

        # Validate output
        result["correct"] = (
            result["output"] == expected_output.strip()
        )

    return result


In [54]:
python_code = """
def find_primes(n):
    primes = []

    for num in range(2, n + 1):
        is_prime = True

        for i in range(2, int(num ** 0.5) + 1):
            if num % i == 0:
                is_prime = False
                break

        if is_prime:
            primes.append(num)

    return primes


print(*find_primes(100))
"""

expected_output = (
    "2 3 5 7 11 13 17 19 23 29 31 37 41 43 "
    "47 53 59 61 67 71 73 79 83 89 97"
)

# Step 1: Generate code
generated_results = generate_all_models(python_code)

# Step 2: Analyze execution time
performance_results = analyze_generated_code(
    generated_results,
    expected_output,
    runs=5
)

# Step 3: Rank models
final_rankings = rank_models(performance_results)


Generating code using Qwen Coder 0.5B...
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 11.2173 seconds
Generation successful: 11.2173s

Generating code using Qwen Coder 1.5B...
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Loading model: Qwen/Qwen2.5-Coder-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 6.9897 seconds
Generation successful: 6.9897s

Generating code using DeepSeek Coder 1.3B...
Loading model: deepseek-ai/deepseek-coder-1.3b-instruct
Loading model: deepseek-ai/deepseek-coder-1.3b-instruct


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 9.7744 seconds
Generation successful: 9.7744s

Generating code using Qwen Coder 3B...
Loading model: Qwen/Qwen2.5-Coder-3B-Instruct
Loading model: Qwen/Qwen2.5-Coder-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully!
Device: cuda:0
Generation completed: 11.1157 seconds
Generation successful: 11.1157s

Analyzing: Qwen Coder 0.5B
Compilation: True
Correctness: True
Execution times: [0.008962156000052346, 0.001687623000179883, 0.0015916509996714012, 0.0016013959998417704, 0.001537188999918726]

Analyzing: Qwen Coder 1.5B
Compilation: False
Correctness: False
Execution times: []

Analyzing: DeepSeek Coder 1.3B
Compilation: True
Correctness: True
Execution times: [0.0019967029998042563, 0.001768682000147237, 0.0016640059998280776, 0.0015579200003230653, 0.001619098999981361]

Analyzing: Qwen Coder 3B
Compilation: True
Correctness: True
Execution times: [0.002084268999624328, 0.0018598169999677339, 0.0017480309998063603, 0.001746551999985968, 0.0016534149999642977]

========== C++ PERFORMANCE ANALYSIS ==========

Rank: 1
Model: Qwen Coder 0.5B
Runs: 5
Average: 0.003076 seconds
Median: 0.001601 seconds
Minimum: 0.001537 seconds
Maximum: 0.008962 seconds
Std deviation: 0.003291 s